In [43]:
!pip install nltk
!pip install torch-geometric
!pip install torch

In [47]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
import torch
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv

# Ensure you have downloaded the required NLTK data
nltk.download('punkt')

# Load data from Excel file
try:
    data = pd.read_excel(r'/content/Data Chatbot STKI.xlsx')
except FileNotFoundError:
    print("File not found!")
    exit()

# Split data into questions and answers
questions = data['QUESTION'].tolist()
answers = data['ANSWER'].tolist()

# Tokenize questions and answers
tokens_questions = [word_tokenize(q.lower()) for q in questions]
tokens_answers = [word_tokenize(a.lower()) for a in answers]

# Create a vocabulary from the dataset
vocab = set()
for q in tokens_questions:
    vocab.update(q)
for a in tokens_answers:
    vocab.update(a)
vocab = {word: idx for idx, word in enumerate(vocab)}

# Create a map from tokenized answers to original answers
tokenized_to_original_answer = {" ".join(word_tokenize(a.lower())): a for a in answers}

# Define a function to create graph data for a single question-answer pair
def create_graph_data(question, answer, vocab):
    edge_index = []
    x = []
    y = []

    question_nodes = list(range(len(x), len(x) + len(question)))
    answer_nodes = list(range(len(x) + len(question), len(x) + len(question) + len(answer)))

    # Create one-hot encoded features for nodes
    for w in question:
        if w in vocab:
            x.append(torch.eye(len(vocab))[vocab[w]])

    for w in answer:
        if w in vocab:
            x.append(torch.eye(len(vocab))[vocab[w]])

    # Create edges for question nodes
    for i in range(len(question_nodes) - 1):
        edge_index.append([question_nodes[i], question_nodes[i + 1]])
        edge_index.append([question_nodes[i + 1], question_nodes[i]])

    # Create edges for answer nodes
    for i in range(len(answer_nodes) - 1):
        edge_index.append([answer_nodes[i], answer_nodes[i + 1]])
        edge_index.append([answer_nodes[i + 1], answer_nodes[i]])

    # Connect the last question node to the first answer node
    if question_nodes and answer_nodes:
        edge_index.append([question_nodes[-1], answer_nodes[0]])
        edge_index.append([answer_nodes[0], question_nodes[-1]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    if x:  # Ensure x is not empty
        x = torch.stack(x)
    else:
        x = torch.empty((0, len(vocab)))  # Handle case when x is empty

    if answer:  # Check if answer is available
        original_answer = tokenized_to_original_answer.get(" ".join(answer), "")
        y = torch.tensor([answers.index(original_answer)] if original_answer else [-1])  # Use the index of the answer as the label
    else:
        original_answer = ""
        y = torch.tensor([-1])  # Set label to -1 if answer is not available

    return Data(x=x, edge_index=edge_index, y=y)

# Create graph data for all question-answer pairs
graph_data_list = [create_graph_data(q, a, vocab) for q, a in zip(tokens_questions, tokens_answers)]


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [56]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import DataLoader
from torch_geometric.nn import GCNConv
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define a GCN model
class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # Apply graph convolution and ReLU activation
        x = self.conv1(x, edge_index)
        x = torch.relu(x)
        x = self.conv2(x, edge_index)

        # Take the mean of node features for graph-level output (optional)
        x = torch.mean(x, dim=0)

        return x

# Parameters
input_dim = len(vocab)
hidden_dim = 64
output_dim = len(answers)

# Create the model, loss function, and optimizer
model = GCN(input_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Create a KFold object for cross-validation
kfold = KFold(n_splits=5, shuffle=True)

# Initialize lists to store test accuracies and losses
test_accuracies = []
test_losses = []
test_precisions = []
test_recalls = []
test_f1_scores = []

# Loop through each fold
for fold, (train_idx, test_idx) in enumerate(kfold.split(graph_data_list)):
    print(f"Fold {fold+1}...")

    # Split the data into training and testing sets
    train_data_list = [graph_data_list[i] for i in train_idx]
    test_data_list = [graph_data_list[i] for i in test_idx]

    # Create DataLoaders
    train_loader = DataLoader(train_data_list, batch_size=1, shuffle=True)
    test_loader = DataLoader(test_data_list, batch_size=1, shuffle=False)

    # Training loop
    num_epochs = 10
    for epoch in range(num_epochs):
        total_loss = 0
        correct = 0
        total = 0
        for data in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output.unsqueeze(0), data.y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(output.data, 0)
            total += data.y.size(0)
            correct += (predicted == data.y).sum().item()

        accuracy = correct / total
        print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {total_loss/len(train_loader)}, Training Accuracy: {accuracy * 100:.2f}%')

    # Test the model
    total_loss = 0
    correct = 0
    total = 0
    y_true = []
    y_pred = []

    with torch.no_grad():
        for data in test_loader:
            output = model(data)
            loss = criterion(output.unsqueeze(0), data.y)
            total_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(output.data, 0)
            total += data.y.size(0)
            correct += (predicted == data.y).sum().item()
            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy().flatten())

    accuracy = correct / total
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    print(f'Fold {fold+1}, Test Loss: {total_loss/len(test_loader)}, Test Accuracy: {accuracy * 100:.2f}%, Test Precision: {precision:.4f}, Test Recall: {recall:.4f}, Test F1 Score: {f1:.4f}')

    # Store test accuracy and loss
    test_accuracies.append(accuracy)
    test_losses.append(total_loss / len(test_loader))
    test_precisions.append(precision)
    test_recalls.append(recall)
    test_f1_scores.append(f1)

print("Training complete.")
print(f"Average Test Accuracy: {sum(test_accuracies) / len(test_accuracies) * 100:.2f}%")
print(f"Average Test Loss: {sum(test_losses) / len(test_losses):.4f}")
print(f"Average Test Precision: {sum(test_precisions) / len(test_precisions):.4f}")
print(f"Average Test Recall: {sum(test_recalls) / len(test_recalls):.4f}")
print(f"Average Test F1 Score: {sum(test_f1_scores) / len(test_f1_scores):.4f}")

Fold 1...


/usr/local/lib/python3.10/dist-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 1/10, Training Loss: 4.054207935565856, Training Accuracy: 0.00%
Epoch 2/10, Training Loss: 3.8873359982560323, Training Accuracy: 2.44%
Epoch 3/10, Training Loss: 3.80250605722753, Training Accuracy: 2.44%
Epoch 4/10, Training Loss: 3.570595723826711, Training Accuracy: 19.51%
Epoch 5/10, Training Loss: 3.1044594398358973, Training Accuracy: 29.27%
Epoch 6/10, Training Loss: 2.36132527124591, Training Accuracy: 48.78%
Epoch 7/10, Training Loss: 1.5641303918347127, Training Accuracy: 73.17%
Epoch 8/10, Training Loss: 0.8639320486747637, Training Accuracy: 92.68%
Epoch 9/10, Training Loss: 0.395378004623259, Training Accuracy: 97.56%
Epoch 10/10, Training Loss: 0.19200576948592576, Training Accuracy: 100.00%
Fold 1, Test Loss: 12.839069799943404, Test Accuracy: 0.00%, Test Precision: 0.0000, Test Recall: 0.0000, Test F1 Score: 0.0000
Fold 2...


/usr/local/lib/python3.10/dist-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 1/10, Training Loss: 2.7391197431105665, Training Accuracy: 73.17%
Epoch 2/10, Training Loss: 1.2744755248380144, Training Accuracy: 73.17%
Epoch 3/10, Training Loss: 0.8387074560699302, Training Accuracy: 78.05%
Epoch 4/10, Training Loss: 0.6850356732654136, Training Accuracy: 75.61%
Epoch 5/10, Training Loss: 0.5176766342142733, Training Accuracy: 85.37%
Epoch 6/10, Training Loss: 0.3780233225240031, Training Accuracy: 92.68%
Epoch 7/10, Training Loss: 0.28953847332244237, Training Accuracy: 92.68%
Epoch 8/10, Training Loss: 0.17349595020286648, Training Accuracy: 97.56%
Epoch 9/10, Training Loss: 0.10767793129538981, Training Accuracy: 100.00%
Epoch 10/10, Training Loss: 0.0609400385150277, Training Accuracy: 100.00%
Fold 2, Test Loss: 0.4479539123448459, Test Accuracy: 100.00%, Test Precision: 1.0000, Test Recall: 1.0000, Test F1 Score: 1.0000
Fold 3...


...

Epoch 1/10, Training Loss: 0.15419651370029896, Training Accuracy: 100.00%
Epoch 2/10, Training Loss: 0.03641597833484411, Training Accuracy: 100.00%
Epoch 3/10, Training Loss: 0.022248514043721593, Training Accuracy: 100.00%
Epoch 4/10, Training Loss: 0.017892032969809537, Training Accuracy: 100.00%
Epoch 5/10, Training Loss: 0.015140051135815503, Training Accuracy: 100.00%
Epoch 6/10, Training Loss: 0.012791604940069928, Training Accuracy: 100.00%
Epoch 7/10, Training Loss: 0.011130915683073303, Training Accuracy: 100.00%
Epoch 8/10, Training Loss: 0.009839607236374701, Training Accuracy: 100.00%
Epoch 9/10, Training Loss: 0.008936016424004697, Training Accuracy: 100.00%
Epoch 10/10, Training Loss: 0.007884143499679686, Training Accuracy: 100.00%
Fold 3, Test Loss: 0.08359822712372988, Test Accuracy: 100.00%, Test Precision: 1.0000, Test Recall: 1.0000, Test F1 Score: 1.0000
Fold 4...


/usr/local/lib/python3.10/dist-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 1/10, Training Loss: 0.03213577268234942, Training Accuracy: 100.00%
Epoch 2/10, Training Loss: 0.009886262964712279, Training Accuracy: 100.00%
Epoch 3/10, Training Loss: 0.008034099458849855, Training Accuracy: 100.00%
Epoch 4/10, Training Loss: 0.007101497016958005, Training Accuracy: 100.00%
Epoch 5/10, Training Loss: 0.00635161948767269, Training Accuracy: 100.00%
Epoch 6/10, Training Loss: 0.005886321800062433, Training Accuracy: 100.00%
Epoch 7/10, Training Loss: 0.005430654614299003, Training Accuracy: 100.00%
Epoch 8/10, Training Loss: 0.0049975496506141056, Training Accuracy: 100.00%
Epoch 9/10, Training Loss: 0.004654450121701562, Training Accuracy: 100.00%
Epoch 10/10, Training Loss: 0.004341360678414016, Training Accuracy: 100.00%
Fold 4, Test Loss: 0.005816454533487558, Test Accuracy: 100.00%, Test Precision: 1.0000, Test Recall: 1.0000, Test F1 Score: 1.0000
Fold 5...


/usr/local/lib/python3.10/dist-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Epoch 1/10, Training Loss: 0.005130776850689601, Training Accuracy: 100.00%
Epoch 2/10, Training Loss: 0.004548840792267583, Training Accuracy: 100.00%
Epoch 3/10, Training Loss: 0.004176855753203632, Training Accuracy: 100.00%
Epoch 4/10, Training Loss: 0.0038643793924988826, Training Accuracy: 100.00%
Epoch 5/10, Training Loss: 0.003580161329453057, Training Accuracy: 100.00%
Epoch 6/10, Training Loss: 0.003385323555059066, Training Accuracy: 100.00%
Epoch 7/10, Training Loss: 0.0031744220524117174, Training Accuracy: 100.00%
Epoch 8/10, Training Loss: 0.003002091773114877, Training Accuracy: 100.00%
Epoch 9/10, Training Loss: 0.002837671283660235, Training Accuracy: 100.00%
Epoch 10/10, Training Loss: 0.0026898640762304978, Training Accuracy: 100.00%
Fold 5, Test Loss: 0.0019366033899132161, Test Accuracy: 100.00%, Test Precision: 1.0000, Test Recall: 1.0000, Test F1 Score: 1.0000
Training complete.
Average Test Accuracy: 80.00%
Average Test Loss: 2.6757
Average Test Precision: 0.80

In [54]:
import torch
import nltk
from nltk.tokenize import word_tokenize
from torch_geometric.data import Data
import pandas as pd

# Load the trained model and necessary vocabularies
# Assuming model, vocab, tokenized_to_original_answer, and answers are already defined and trained

# Function to preprocess and tokenize the query
def preprocess_query(query, vocab):
    tokens_query = word_tokenize(query.lower())
    x = []

    # Create one-hot encoded features for nodes
    for w in tokens_query:
        if w in vocab:
            x.append(torch.eye(len(vocab))[vocab[w]])

    # Handle case when x is empty
    if x:
        x = torch.stack(x)
    else:
        x = torch.empty((0, len(vocab)))

    return x

# Function to get predicted answer for a query
def get_predicted_answer(query, model, vocab, tokenized_to_original_answer):
    # Preprocess the query
    x_query = preprocess_query(query, vocab)

    # Create dummy edge_index (since it's not used in inference)
    edge_index = torch.empty((2, 0), dtype=torch.long)

    # Create Data object
    data = Data(x=x_query, edge_index=edge_index)

    # Perform inference
    with torch.no_grad():
        output = model(data)
        _, predicted_idx = torch.max(output, 0)

        # Map predicted index to original answer
        predicted_answer_idx = predicted_idx.item()
        # predicted_answer = tokenized_to_original_answer[answers[predicted_answer_idx]]
        predicted_answer = answers[predicted_answer_idx]


    return predicted_answer

# Example usage:
query = input("Search Here :")
real_answer = get_predicted_answer(query, model, vocab, tokenized_to_original_answer)
print(f"Predicted Answer: {real_answer}")

Search Here :jelaskan informasi mengenai bu Amilah!
Predicted Answer: NIK : 199210262020013201\nNama : Amila Sofiah, S.T., M.T.\nPendidikan : S2 Teknik Elektro, Institut Teknologi Bandung\nResearch Interest : Biomedical Signal Processing, Instrumentation & Control, Robotics\nEmail : amila.sofiah@ftmm.unair.ac.id
